In [1]:
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns
import pandas            as pd
import SO_library        as sol
import json
import os

# Assuming the `bin` directory is in the user's home directory
# For sqs
home_directory = os.path.expanduser("~")
bin_directory = os.path.join(home_directory, "bin")
os.environ["PATH"] += os.pathsep + bin_directory

from pymatgen.core.surface   import generate_all_slabs
from pymatgen.core.structure import Structure
from pymatgen.io.ase         import AseAtomsAdaptor
from ase.io.vasp             import write_vasp

sns.set_theme()

/home/claudio/cibran/Work/UPC/SlabOptimization/venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_in

We only consider coherent interfaces (which has matching lattices from both sides of the interface, meaning that there is a repeat along the interface that lets build periodicity along the interface surface).

In [2]:
# Define name of folder and path to reference POSCAR
general_folder = 'input/BiSeI'
path_to_POSCAR = 'POSCAR-uc-BiSeI'

# Maximum number for the Miller index in each direction
max_index = 2

# Whether to repair terminations with broken bonds or just omit them
repair = True

# Minimum thicknesses for slab and vacuum
min_slab_size   = 20.0
min_vacuum_size = 20.0

# Define paths to pretrained model and structure to be relaxed
# Materials Project pretrained model as default
#model_load_path = '../../UCL/m3gnet/finetuned_model'
model_load_path = None
model_load_path = 'large' if model_load_path is None else model_load_path
model_load_path

'large'

In [3]:
# Move POSCAR to defect folder
# The folder is named as defects_x_y_z_vi (eg, defects_0_1_0.5_v0)
# Each new run generates a new folder, with different defects most likely (as POSCAR will vary as well)
i = 0
while True:
    if not os.path.exists(general_folder):
        # Generate new folder
        os.system(f'mkdir {general_folder}')
    
    slab_folder = f'{general_folder}/slab_v{i}'
    if not os.path.exists(slab_folder):
        # Generate new folder
        os.system(f'mkdir {slab_folder}')

        # Copy POSCAR there (named as unticell, and POSCAR for creating supercell)
        os.system(f'cp {path_to_POSCAR} {slab_folder}/POSCAR')
        break
    i +=1
slab_folder

'input/BiSeI/slab_v0'

# ML-IAP relaxation

In [4]:
# Unit-cell relaxation

# Define paths to bulk and POSCAR
path_to_bulk = f'{slab_folder}/bulk'

# Generate directory for current slab
os.system(f'mkdir {path_to_bulk}')

# Copy POSCAR to new directory
os.system(f'cp {slab_folder}/POSCAR {path_to_bulk}/POSCAR')

# Relax the structure, hiding the output (verbose=False)
_ = sol.structural_relaxation(f'{path_to_bulk}/POSCAR',
                              model_load_path,
                              output_folder=path_to_bulk)

# Get the single-shot energy
bulk_energy, _, _ = sol.single_shot_energy_calculation(f'{path_to_bulk}/CONTCAR',
                                                       model_load_path)

# Read relaxed structure
structure = Structure.from_file(f'{path_to_bulk}/CONTCAR')

# Save slab information
slab_data = {
    'number_of_sites': structure.num_sites,
}
with open(f'{path_to_bulk}/slab_data.json', 'w') as json_file:
    json.dump(slab_data, json_file)

Using Materials Project MACE for MACECalculator with /home/claudio/.cache/mace/MACE_MPtrj_20229model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/home/claudio/cibran/Work/UPC/SlabOptimization/venv/lib/python3.12/site-packages/mace/calculators/mace.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=mod

      Step     Time          Energy          fmax
BFGS:    0 11:47:30      -39.968874       10.600288
BFGS:    1 11:47:31      -35.587339       26.649613
BFGS:    2 11:47:31      -40.252449        6.516512
BFGS:    3 11:47:32      -40.293648        6.581378
BFGS:    4 11:47:32      -40.427194        0.413472
BFGS:    5 11:47:32      -40.446715        0.389916
BFGS:    6 11:47:32      -40.550978        0.272335
BFGS:    7 11:47:32      -40.558595        0.200009
BFGS:    8 11:47:32      -40.567890        0.172324
BFGS:    9 11:47:32      -40.596642        0.194716
BFGS:   10 11:47:32      -40.608276        0.231934
BFGS:   11 11:47:33      -40.614793        0.163712
BFGS:   12 11:47:33      -40.619831        0.177513
BFGS:   13 11:47:33      -40.634331        0.308144
BFGS:   14 11:47:33      -40.656812        2.191954
BFGS:   15 11:47:33      -40.695681        0.743145
BFGS:   16 11:47:33      -40.730077        0.554213
BFGS:   17 11:47:33      -40.782165        1.104562
BFGS:   18 11:

/home/claudio/cibran/Work/UPC/SlabOptimization/venv/lib/python3.12/site-packages/mace/calculators/mace.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=mod

# Slab generation

In [5]:
# Generate all slabs
slabs = generate_all_slabs(structure,
                           max_index=max_index,
                           min_slab_size=min_slab_size,
                           min_vacuum_size=min_vacuum_size,
                           repair=repair)

# Get energy per atom
bulk_energy_per_atom = bulk_energy / structure.num_sites

In [6]:
for n, slab in enumerate(slabs):
    print(n, "Polar:", slab.is_polar(), "Symmetric: ", slab.is_symmetric())

0 Polar: False Symmetric:  True
1 Polar: False Symmetric:  False
2 Polar: False Symmetric:  False
3 Polar: False Symmetric:  False
4 Polar: False Symmetric:  False
5 Polar: False Symmetric:  True
6 Polar: False Symmetric:  False
7 Polar: False Symmetric:  False
8 Polar: False Symmetric:  True
9 Polar: False Symmetric:  True
10 Polar: False Symmetric:  False
11 Polar: False Symmetric:  False
12 Polar: False Symmetric:  True
13 Polar: False Symmetric:  True
14 Polar: False Symmetric:  True
15 Polar: False Symmetric:  False
16 Polar: False Symmetric:  False
17 Polar: False Symmetric:  False
18 Polar: False Symmetric:  False
19 Polar: False Symmetric:  True
20 Polar: False Symmetric:  True
21 Polar: False Symmetric:  True
22 Polar: False Symmetric:  False
23 Polar: False Symmetric:  False
24 Polar: False Symmetric:  True
25 Polar: False Symmetric:  True
26 Polar: False Symmetric:  False
27 Polar: False Symmetric:  False
28 Polar: False Symmetric:  True
29 Polar: False Symmetric:  False
30 

Surface formation energy of shape $S$ for slab of energy $E_S$ with $N$ formula units and $E_{bulk}$ bulk energy is:

\begin{equation}
    E_S = \frac{E_S - E_{bulk}}{2 S}
\end{equation}

where all energies are per atom.

In [ ]:
# Initialize the data dictionary for storing all slab energy calculations
# Iterate over slabs
slab_energies = {}
for i, slab in enumerate(slabs):
    print()
    print(f'Slab {i+1}')
    print()
    print('Miller index:', slab.miller_index)
    print('Shift:', slab.shift)
    print('Surface area:', slab.surface_area)
    print('Number of sites:', len(slab.sites))

    # Miller index tuple to string
    miller_index_str = '_'.join(str(element) for element in slab.miller_index)
    miller_index_str = f'{miller_index_str}_i_{i}'

    # Define current folder
    current_folder = f'{slab_folder}/{miller_index_str}'

    # Generate new folder
    os.system(f'mkdir {current_folder}')

    # Save slab into current_folder
    write_vasp(f'{current_folder}/POSCAR', AseAtomsAdaptor.get_atoms(slab), direct=True, sort=True)

    # Save slab information
    slab_data = {
        'miller_index': slab.miller_index,
        'shift': slab.shift,
        'surface_area': slab.surface_area,
        'number_of_sites': len(slab.sites),
        'is_polar': str(slab.is_polar()),
        'is_symmetric': str(slab.is_symmetric())
    }
    with open(f'{current_folder}/slab_data.json', 'w') as json_file:
        json.dump(slab_data, json_file)

    # Relax the structure, hiding the output (verbose=False)
    _ = sol.structural_relaxation(f'{current_folder}/POSCAR',
                                  model_load_path,
                                  relax_cell=False,
                                  output_folder=current_folder)

    # Get and save the single-shot energy
    ssc_energy, _, _ = sol.single_shot_energy_calculation(f'{current_folder}/CONTCAR',
                                                          model_load_path)
    np.savetxt(f'{current_folder}/single_shot_energy', [ssc_energy])

    # Get energy per atom
    ssc_energy_per_atom = ssc_energy / len(slab.sites)

    # Compute slab energy in eV/atom/ang^2
    slab_energy = (ssc_energy_per_atom - bulk_energy_per_atom) / (2 * slab.surface_area)

    print(f'\t{slab_energy} eV/atom/Å^2')

    # Generate a dictionary object with the new data and update in the main data object
    slab_energies.update({
        (miller_index_str): slab_energy
    })

# Convert to Pandas DataFrame
slab_energies = pd.DataFrame(slab_energies, index=['energy'])


Slab 1

Miller index: (1, 1, 1)
Shift: 6.350475700855895e-14
Surface area: 128.70698542845744
Number of sites: 72
Using Materials Project MACE for MACECalculator with /home/claudio/.cache/mace/MACE_MPtrj_20229model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/home/claudio/cibran/Work/UPC/SlabOptimization/venv/lib/python3.12/site-packages/mace/calculators/mace.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=mod

      Step     Time          Energy          fmax
BFGS:    0 11:48:08     -241.447633        1.130303
BFGS:    1 11:48:08     -241.602370        1.036118
BFGS:    2 11:48:08     -242.169759        1.494758
BFGS:    3 11:48:08     -242.322650        0.740542
BFGS:    4 11:48:08     -242.395489        0.705098
BFGS:    5 11:48:08     -242.710887        0.621265
BFGS:    6 11:48:08     -242.750178        0.402060
BFGS:    7 11:48:09     -242.785110        0.286893
BFGS:    8 11:48:09     -242.881130        0.351119
BFGS:    9 11:48:09     -242.935721        0.453066
BFGS:   10 11:48:09     -242.982582        0.491603
BFGS:   11 11:48:09     -243.033218        0.466419
BFGS:   12 11:48:09     -243.109815        0.486753
BFGS:   13 11:48:09     -243.179353        0.395109
BFGS:   14 11:48:09     -243.219170        0.307970
BFGS:   15 11:48:09     -243.246467        0.273990
BFGS:   16 11:48:10     -243.269141        0.234654
BFGS:   17 11:48:10     -243.298022        0.305756
BFGS:   18 11:

# Extract local minimas and generate input files

In [ ]:
# Sorted in ascendent order
min_arg       = np.argsort(slab_energies.values)[0]
local_minimas = slab_energies.columns[min_arg]
energies      = slab_energies.values[0][min_arg]

In [ ]:
# Energy differences in eV/supercell
plt.figure(figsize=(15, 5))
plt.plot(energies, 'o-')
plt.xticks(range(len(local_minimas)), local_minimas, rotation='vertical')
plt.ylabel(r'$\Delta E$ (eV/atom/Å^2)')
plt.savefig(f'{slab_folder}/ranking.eps', dpi=50, bbox_inches='tight')
plt.show()